In [11]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)
    print(data["publications"])

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "amici2012aversion")
original_data_pathway = os.path.join(pathway, "original_data")
complete_path_1 = os.path.join(original_data_pathway, "Amici_2012_exp1_Behaviour_AVES.csv")
complete_path_2 = os.path.join(original_data_pathway, "Amici_2012_exp_2_Behaviour_AVES.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)


../study


In [12]:
import pandas as pd
import numpy as np

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, 'amici2012aversion', 'social_tolerance_task', '1'],
                        [df2, 'amici2012aversion', 'food_distribution', '2']]

for x,y,k,b in experiment_import:
    x['study_id']=y
    x['experiment_name']=k
    x['experiment']=b

# df1.columns

In [13]:
# pathway_gen = os.path.join(data["python_files"])
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x['subject'] = x['subject'].str.rstrip()
    x['partner'] = x['partner'].str.rstrip()
    x=x.rename(columns={"subject": "ape", "partner": "ape_2"})
    data_frames[index]=x
new_df1=data_frames[0]
new_df2=data_frames[1]

fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [14]:
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)
fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')

In [15]:
fulldf['role']="focal_participant"
fulldf['role_2']="partner"

In [16]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [17]:
# fulldf.columns

In [18]:
fulldf['ape'].replace('', np.nan, inplace=True)
fulldf.dropna(subset=['ape'], inplace=True)


In [19]:
fulldf = fulldf.rename(columns={"species_y": "species", 
        "drinking time (ss)": "drinking_time_in_seconds",
        "codrinking time (ss)": "codrinking_time_in_seconds",
        "session duration (ss)": "session_duration_in_seconds",
        "proximity (ss)": "proximity_in_seconds",
        "composite index (ss)": "composite_index_in_seconds",
        "begging (ss)": "begging_in_seconds",
        "time (ss)": "time_in_seconds"})


fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)
fulldf['year']=fulldf['year'].astype(int)
fulldf['month']=fulldf['month'].astype(int)
fulldf['day']=fulldf['day'].astype(int)

In [20]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
fulldf= fulldf.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
fulldf['dodc_2'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc_2'] = pd.to_datetime(fulldf['dodc_2'])
fulldf['dob_2'] = pd.to_datetime(fulldf['dob_2'])

fulldf['age_in_years_2'] = (fulldf['dodc_2'] - fulldf['dob_2']).dt.days//365

In [21]:
fulldf['condition'].replace(' - ', '-', inplace=True, regex=True)
fulldf['condition'].replace(' ', '_', inplace=True, regex=True) 

In [22]:
fulldf=fulldf[['study_id', 'experiment','experiment_name', 'year', 'month', 'day', 
               'participant',  'age_in_years','sex',
        'role', 'participant_2', 'age_in_years_2', 'sex_2',  'role_2','species', 'dyad',
       'condition', 'drinking_time_in_seconds', 'codrinking_time_in_seconds',
       'session_duration_in_seconds', 'proximity_in_seconds',
       'composite_index_in_seconds', 'begging_in_seconds', 'time_in_seconds']]

In [23]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'amici2012aversion_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'amici2012aversion_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)